# v6-DA — Phase 2 Inference Notebook (Per-Image Normalization)

**Model**: `DualSwinFusionSeg` — Dual Swin V2 Small, Hybrid SegFormer×UNet++ decoder

**Key change vs v6**: Per-image percentile normalization (domain-invariant).
Each band is clipped to its own image's P1/P99 → [0,1], then z-scored with global mean/std.

**Attention**: `input_attn="eca"`, `intra_encoder_attn="se"`, `decoder_output_attn="cbam"`

**Usage**:
1. Set `TEST_DATA_DIR` to phase2 test folder with `.tif` files
2. Set `CHECKPOINT_BASE_DIR` to folder with checkpoint subdir
3. Set `STATS_JSON_PATH` to the `norm_stats_v6_DA.json` produced by v6-DA training
4. Run all cells → outputs `phase2_submission_DA.zip`

In [ ]:
# ============================================================
# Cell 1 — Imports, device, determinism
# ============================================================
import os, json, math, zipfile, warnings, random
from pathlib import Path

import numpy as np
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import rasterio
import timm

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name()}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'  cudnn.deterministic = {torch.backends.cudnn.deterministic}')

In [ ]:
# ============================================================
# Cell 2 — ★ CONFIGURABLE PATHS & SETTINGS ★
# ============================================================

# --- Checkpoint base directory ---
# v6-DA training saves to: kfold_results_v6_archive_DA/checkpoints/{tag}/
CHECKPOINT_BASE_DIR = "/kaggle/input/datasets/iftekharabid/ph1-86-86-da/kfold_results_v6_p1-86_DA/checkpoints"  # ★ CHANGE

# --- Phase 2 test data (folder with *.tif files) ---
# Phase 1 test:  TEST_DATA_DIR = "data/phase1_dataset/test/images"
# Phase 2 test:  TEST_DATA_DIR = "data/phase2_dataset/test/images"
TEST_DATA_DIR = "data/phase2_dataset/test/images"  # ★ CHANGE

# --- Output ---
OUTPUT_DIR = "phase2_86.86_inference_DA_output"

# --- Normalization stats (saved by v6-DA training) ---
# Contains channel_means & channel_stds computed after per-image normalization.
STATS_JSON_PATH = "/kaggle/input/datasets/amimulehsan1/v6-da-output/kfold_results_v6_archive_DA/norm_stats_v6_DA.json"  # ★ CHANGE
# If STATS_JSON_PATH is None, recompute from archive data (slow):
ARCHIVE_DATA_DIR = "data/phase1_dataset"  # only used when STATS_JSON_PATH is None

# --- Model / inference config (MUST match training cfg exactly) ---
ENCODER_NAME        = "swinv2_small_window8_256"
DECODER_NAME        = "hybrid_segformer_unetpp"
FUSIONS             = ["concat_eca"]
INPUT_ATTN          = "eca"
POST_ENCODER_ATTN   = "none"
INTRA_ENCODER_ATTN  = "se"
DECODER_OUTPUT_ATTN = "cbam"
IMG_SIZE            = 128        # ★ native resolution
ORIG_SIZE           = 128
FPN_CHANNELS        = 256
BATCH_SIZE          = 8
NUM_WORKERS         = 2
THRESH              = 0.5
USE_TTA             = True
NUM_FOLDS           = 5

# --- Channel config (must match training) ---
RGB_INDICES  = [4, 5, 6]   # B4, B5, B6 (0-indexed in 7-band array)
AUX_INDICES  = [0, 1, 2, 3]  # CTX, SLOPE, DEM, B3
BAND_INDICES_ALL = list(range(1, 8))  # rasterio 1-based bands 1-7

# Resolve dirs
CHECKPOINT_BASE_DIR = Path(CHECKPOINT_BASE_DIR)
TEST_DATA_DIR       = Path(TEST_DATA_DIR)
OUTPUT_DIR          = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Build checkpoint folder name exactly like training ---
def _build_ckpt_tag(fusion_name):
    attn_tag = ""
    if INPUT_ATTN != "none":
        attn_tag += f"_inA{INPUT_ATTN}"
    if POST_ENCODER_ATTN != "none":
        attn_tag += f"_peA{POST_ENCODER_ATTN}"
    if INTRA_ENCODER_ATTN != "none":
        attn_tag += f"_ieA{INTRA_ENCODER_ATTN}"
    if DECODER_OUTPUT_ATTN != "none":
        attn_tag += f"_doA{DECODER_OUTPUT_ATTN}"
    return f"{ENCODER_NAME.split('_')[0]}_{DECODER_NAME}_{fusion_name}{attn_tag}"

CKPT_DIRS = {
    f: CHECKPOINT_BASE_DIR / _build_ckpt_tag(f)
    for f in FUSIONS
}

print(f'Checkpoint base: {CHECKPOINT_BASE_DIR}')
for f, d in CKPT_DIRS.items():
    print(f'  {f}: {d}')
print(f'Test data:       {TEST_DATA_DIR}')
print(f'Output:          {OUTPUT_DIR}')
print(f'Stats JSON:      {STATS_JSON_PATH}')
print(f'IMG_SIZE={IMG_SIZE}  ORIG_SIZE={ORIG_SIZE}  TTA={USE_TTA}  Thresh={THRESH}  Folds={NUM_FOLDS}')
print(f'Fusions to ensemble: {FUSIONS}')

print(f'Attention: input={INPUT_ATTN}  post_enc={POST_ENCODER_ATTN}  intra_enc={INTRA_ENCODER_ATTN}  dec_out={DECODER_OUTPUT_ATTN}')print(f'★ v6-DA: Per-image percentile normalization (domain-invariant)')

In [ ]:
# ============================================================
# Cell 3 — ★ v6-DA: Per-Image Normalization + Load Stats from JSON
# ============================================================

def normalize_bands_per_image(arr_chw, p_low=1.0, p_high=99.0):
    """Per-image, per-band percentile normalization — domain-invariant.

    Each band is clipped to its OWN image's [P_low, P_high] percentiles and
    rescaled to [0, 1].  This eliminates sensitivity to absolute value shifts
    between domains (e.g. DEM at different elevations).
    """
    C = arr_chw.shape[0]
    out = np.empty_like(arr_chw, dtype=np.float32)
    for c in range(C):
        flat = arr_chw[c].ravel()
        lo = np.percentile(flat, p_low)
        hi = np.percentile(flat, p_high)
        hi = max(hi, lo + 1e-6)
        out[c] = np.clip((arr_chw[c].astype(np.float32) - lo) / (hi - lo), 0.0, 1.0)
    return out


def compute_mean_std_per_image_norm(img_paths, band_indices,
                                     p_low=1.0, p_high=99.0,
                                     max_files=None, max_pixels=2_000_000):
    """Compute channel mean/std AFTER per-image percentile normalization."""
    paths = list(img_paths)
    if max_files is not None:
        paths = paths[:max_files]
    C = len(band_indices)
    sums = np.zeros(C, dtype=np.float64)
    sqs  = np.zeros(C, dtype=np.float64)
    n    = 0
    rng  = np.random.default_rng(123)
    for p in tqdm(paths, desc="Compute mean/std (per-image norm)"):
        with rasterio.open(str(p)) as src:
            arr = src.read(band_indices).astype(np.float32)
        arr = normalize_bands_per_image(arr, p_low, p_high)
        flat = arr.reshape(C, -1)
        if flat.shape[1] > 20000:
            idx = rng.choice(flat.shape[1], size=20000, replace=False)
            flat = flat[:, idx]
        sums += flat.sum(axis=1)
        sqs  += (flat * flat).sum(axis=1)
        n    += flat.shape[1]
        if n >= max_pixels:
            break
    means = (sums / max(n, 1)).astype(np.float32)
    vars_ = (sqs  / max(n, 1) - means.astype(np.float64) ** 2).clip(min=1e-8)
    stds  = np.sqrt(vars_).astype(np.float32)
    return means, stds


# ------- Load stats from JSON or recompute -------
if STATS_JSON_PATH and Path(STATS_JSON_PATH).exists():
    with open(STATS_JSON_PATH) as f:
        _s = json.load(f)
    MEANS = np.array(_s['channel_means'], dtype=np.float32)
    STDS  = np.array(_s['channel_stds'],  dtype=np.float32)
    print(f'Loaded per-image norm stats from {STATS_JSON_PATH}')
    print(f'  normalization: {_s.get("normalization", "unknown")}')
    print(f'  p_low={_s.get("p_low", "?")}, p_high={_s.get("p_high", "?")}')
else:
    print(f'STATS_JSON_PATH not found — recomputing from {ARCHIVE_DATA_DIR}')
    _root = Path(ARCHIVE_DATA_DIR)
    _all_imgs = sorted(
        list((_root / 'train' / 'images').glob('*.tif')) +
        list((_root / 'val'   / 'images').glob('*.tif'))
    )
    print(f'  Found {len(_all_imgs)} train+val images for stats computation')
    assert len(_all_imgs) > 0, f'No .tif images found!'
    MEANS, STDS = compute_mean_std_per_image_norm(_all_imgs, BAND_INDICES_ALL)
    # Save for next time
    _save = {
        'normalization': 'per_image_percentile',
        'p_low': 1.0, 'p_high': 99.0,
        'band_indices_all': BAND_INDICES_ALL,
        'rgb_indices': RGB_INDICES, 'aux_indices': AUX_INDICES,
        'channel_means': MEANS.tolist(), 'channel_stds': STDS.tolist(),
        'img_size': IMG_SIZE,
    }
    _out_stats = OUTPUT_DIR / 'norm_stats_v6_DA.json'
    _out_stats.write_text(json.dumps(_save, indent=2))
    print(f'  Stats saved → {_out_stats}')

print(f'\n★ v6-DA Per-Image Normalization Stats ({len(BAND_INDICES_ALL)} channels):')
print(f'  mean = {MEANS}')
print(f'  std  = {STDS}')
print(f'  (No global low/high — each test image uses its own P1/P99)')

In [ ]:
# ============================================================
# Cell 4 — Full Model Architecture
# (Hybrid SegFormer×UNet++ decoder, ECA/CBAM fusions,
#  SwinWithIntraAttention, DualSwinFusionSeg)
# ============================================================

# ---- Backbone helpers ----
def adapt_patch_embed_in_chans(model, in_chans_new):
    pe = model.patch_embed
    old_conv = pe.proj
    old_w = old_conv.weight.data
    embed_dim, old_in, kh, kw = old_w.shape
    new_conv = nn.Conv2d(
        in_chans_new, embed_dim,
        kernel_size=old_conv.kernel_size, stride=old_conv.stride,
        padding=old_conv.padding, bias=(old_conv.bias is not None),
    )
    with torch.no_grad():
        new_w = torch.zeros(embed_dim, in_chans_new, kh, kw, device=old_w.device)
        new_w[:, :3, :, :] = old_w
        if in_chans_new > 3:
            rgb_mean = old_w.mean(dim=1, keepdim=True)
            new_w[:, 3:, :, :] = rgb_mean.repeat(1, in_chans_new - 3, 1, 1)
        new_conv.weight.copy_(new_w)
        if old_conv.bias is not None:
            new_conv.bias.copy_(old_conv.bias.data)
    pe.proj = new_conv
    return model

def make_swin_features(encoder_name, pretrained=True, img_size=256):
    enc = timm.create_model(
        encoder_name, pretrained=pretrained,
        features_only=True, out_indices=(0, 1, 2, 3), img_size=img_size,
    )
    if hasattr(enc, 'patch_embed'):
        enc.patch_embed.img_size = None
        if hasattr(enc.patch_embed, 'strict_img_size'):
            enc.patch_embed.strict_img_size = False
    return enc

def to_nchw(feats, in_chs):
    out = []
    for f, c in zip(feats, in_chs):
        if f.ndim == 4 and f.shape[-1] == c and f.shape[1] != c:
            f = f.permute(0, 3, 1, 2).contiguous()
        out.append(f)
    return out

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class SEBlock(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        mid = max(ch // r, 8)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, mid, 1), nn.ReLU(inplace=True),
            nn.Conv2d(mid, ch, 1), nn.Sigmoid(),
        )
    def forward(self, x): return x * self.se(x)

# ---- Channel Attention Modules ----
class SE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, mid, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False), nn.Sigmoid(),
        )
    def forward(self, x): return x * self.fc(x)

class ECA(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        y = self.avg_pool(x).squeeze(-1).transpose(-1, -2)
        y = self.conv(y).transpose(-1, -2).unsqueeze(-1)
        return x * self.sigmoid(y)

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False),
            nn.Sigmoid(),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.mlp(F.adaptive_avg_pool2d(x, 1))
        max_out = self.mlp(F.adaptive_max_pool2d(x, 1))
        x = x * self.sigmoid(avg_out + max_out)
        avg_s = x.mean(dim=1, keepdim=True)
        max_s, _ = x.max(dim=1, keepdim=True)
        return x * self.spatial(torch.cat([avg_s, max_s], dim=1))

# ---- Attention strategy modules ----
def _make_attn(attn_type, channels, reduction=16):
    t = attn_type.lower()
    if t == 'none':  return nn.Identity()
    if t == 'se':    return SE(channels, reduction)
    if t == 'eca':   return ECA(channels)
    if t == 'cbam':  return CBAM(channels, reduction)
    raise ValueError(f'Unknown attention type: {attn_type}')

class InputChannelAttention(nn.Module):
    def __init__(self, in_channels, attn_type='none', reduction=16):
        super().__init__()
        self.attn = _make_attn(attn_type, in_channels, reduction)
    def forward(self, x): return self.attn(x)

class PostEncoderAttention(nn.Module):
    def __init__(self, channel_list, attn_type='none', reduction=16):
        super().__init__()
        self.attns = nn.ModuleList([_make_attn(attn_type, c, reduction) for c in channel_list])
    def forward(self, feats):
        return [self.attns[i](f) for i, f in enumerate(feats)]

class DecoderOutputAttention(nn.Module):
    def __init__(self, channels, attn_type='none', reduction=16):
        super().__init__()
        self.attn = _make_attn(attn_type, channels, reduction)
    def forward(self, x): return self.attn(x)

class SwinWithIntraAttention(nn.Module):
    """Wraps Swin encoder and injects channel attention after each stage."""
    def __init__(self, base_encoder, attn_type='none', reduction=16):
        super().__init__()
        self.base_encoder = base_encoder
        self.attn_type = attn_type.lower()
        self.feature_info = base_encoder.feature_info
        self.chs = base_encoder.feature_info.channels()
        if self.attn_type == 'none':
            self.stage_attns = nn.ModuleList([nn.Identity() for _ in self.chs])
        else:
            self.stage_attns = nn.ModuleList([_make_attn(attn_type, c, reduction) for c in self.chs])
        self._hook_handles = []
        self._register_hooks()

    def _get_stage_modules(self):
        if hasattr(self.base_encoder, 'stages'):
            return list(self.base_encoder.stages)
        if hasattr(self.base_encoder, 'layers'):
            return list(self.base_encoder.layers)
        return []

    def _register_hooks(self):
        if self.attn_type == 'none':
            return
        for i, stage in enumerate(self._get_stage_modules()):
            handle = stage.register_forward_hook(self._make_hook(i))
            self._hook_handles.append(handle)

    def _make_hook(self, stage_idx):
        def hook(module, input, output):
            if output.ndim == 4:
                c = self.chs[stage_idx]
                if output.shape[-1] == c and output.shape[1] != c:
                    out_nchw = output.permute(0, 3, 1, 2).contiguous()
                    out_attn = self.stage_attns[stage_idx](out_nchw)
                    output = out_attn.permute(0, 2, 3, 1).contiguous()
                else:
                    output = self.stage_attns[stage_idx](output)
            return output
        return hook

    def remove_hooks(self):
        for h in self._hook_handles: h.remove()
        self._hook_handles = []

    def forward(self, x): return self.base_encoder(x)

    def __getattr__(self, name):
        if name in ['base_encoder', 'attn_type', 'chs', 'stage_attns',
                    '_hook_handles', 'feature_info']:
            return super().__getattr__(name)
        return getattr(self.base_encoder, name)

def make_swin_with_intra_attention(encoder_name, pretrained=True, img_size=256,
                                   intra_attn='none', reduction=16):
    base_enc = make_swin_features(encoder_name, pretrained=pretrained, img_size=img_size)
    if intra_attn.lower() == 'none':
        return base_enc
    return SwinWithIntraAttention(base_enc, attn_type=intra_attn, reduction=reduction)

# ---- Hybrid SegFormer×UNet++ Decoder ----
class HybridSegFormerUNetPPDecoder(nn.Module):
    def __init__(self, in_channels_list, fpn_channels=256):
        super().__init__()
        C = fpn_channels
        self.mlp_projs = nn.ModuleList([
            nn.Sequential(nn.Conv2d(c, C, 1, bias=False), nn.BatchNorm2d(C), nn.ReLU(inplace=True))
            for c in in_channels_list
        ])
        self.seg_fuse = nn.Sequential(
            nn.Conv2d(C * 4, C, 1, bias=False), nn.BatchNorm2d(C),
            nn.ReLU(inplace=True), nn.Dropout2d(0.1),
        )
        def _node(n_cat):
            return nn.Sequential(ConvBNReLU(C * n_cat, C), ConvBNReLU(C, C))
        self.x01 = _node(2); self.x11 = _node(2); self.x21 = _node(2)
        self.x02 = _node(3); self.x12 = _node(3)
        self.x03 = _node(4)
        self.upp_final = nn.Sequential(ConvBNReLU(C, C), nn.Dropout2d(0.1))
        self.gate_fuse = nn.Sequential(
            nn.Conv2d(2 * C, C, 1, bias=False), nn.BatchNorm2d(C), nn.ReLU(inplace=True))
        self.gate_se = SEBlock(C, r=16)
        self.aux_head_01 = nn.Conv2d(C, 1, 1)
        self.aux_head_02 = nn.Conv2d(C, 1, 1)

    @staticmethod
    def _up(x, t):
        return F.interpolate(x, size=t.shape[-2:], mode='bilinear', align_corners=False)

    def forward(self, feats):
        x00 = self.mlp_projs[0](feats[0])
        x10 = self.mlp_projs[1](feats[1])
        x20 = self.mlp_projs[2](feats[2])
        x30 = self.mlp_projs[3](feats[3])
        target = x00.shape[-2:]
        seg_global = self.seg_fuse(torch.cat([
            x00,
            F.interpolate(x10, size=target, mode='bilinear', align_corners=False),
            F.interpolate(x20, size=target, mode='bilinear', align_corners=False),
            F.interpolate(x30, size=target, mode='bilinear', align_corners=False),
        ], dim=1))
        x21 = self.x21(torch.cat([x20, self._up(x30, x20)], dim=1))
        x11 = self.x11(torch.cat([x10, self._up(x20, x10)], dim=1))
        x01 = self.x01(torch.cat([x00, self._up(x10, x00)], dim=1))
        x12 = self.x12(torch.cat([x10, x11, self._up(x21, x10)], dim=1))
        x02 = self.x02(torch.cat([x00, x01, self._up(x11, x00)], dim=1))
        x03 = self.x03(torch.cat([x00, x01, x02, self._up(x12, x00)], dim=1))
        upp_out = self.upp_final(x03)
        fused = self.gate_se(self.gate_fuse(torch.cat([upp_out, seg_global], dim=1)))
        aux_list = [self.aux_head_01(x01), self.aux_head_02(x02)]
        return fused, aux_list

# ---- Fusion strategies ----
class FusionConcatECA(nn.Module):
    def __init__(self, chs):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Conv2d(2*c, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
            for c in chs
        ])
        self.eca = nn.ModuleList([ECA(c) for c in chs])
    def forward(self, A, B):
        return [self.eca[i](self.proj[i](torch.cat([a, b], dim=1)))
                for i, (a, b) in enumerate(zip(A, B))]

class FusionConcatCBAM(nn.Module):
    def __init__(self, chs, r=16):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Conv2d(2*c, c, 1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
            for c in chs
        ])
        self.cbam = nn.ModuleList([CBAM(c, r) for c in chs])
    def forward(self, A, B):
        return [self.cbam[i](self.proj[i](torch.cat([a, b], dim=1)))
                for i, (a, b) in enumerate(zip(A, B))]

def build_fusion(name, chs):
    name = name.lower()
    if name == 'concat_eca':  return FusionConcatECA(chs)
    if name == 'concat_cbam': return FusionConcatCBAM(chs)
    raise ValueError(f'Unsupported fusion for inference: {name}')

# ---- Main model ----
class DualSwinFusionSeg(nn.Module):
    def __init__(self, encoder_name, pretrained, img_size, fpn_channels, fusion_name,
                 decoder_name='hybrid_segformer_unetpp',
                 input_attn='none', post_encoder_attn='none',
                 intra_encoder_attn='none', decoder_output_attn='none'):
        super().__init__()
        self.enc_rgb = make_swin_with_intra_attention(
            encoder_name, pretrained=pretrained, img_size=img_size, intra_attn=intra_encoder_attn)
        self.enc_aux = make_swin_with_intra_attention(
            encoder_name, pretrained=pretrained, img_size=img_size, intra_attn=intra_encoder_attn)
        if intra_encoder_attn.lower() == 'none':
            adapt_patch_embed_in_chans(self.enc_aux, 4)
        else:
            adapt_patch_embed_in_chans(self.enc_aux.base_encoder, 4)
        self.chs = self.enc_rgb.feature_info.channels()
        self.rgb_input_attn   = InputChannelAttention(3, attn_type=input_attn)
        self.aux_input_attn   = InputChannelAttention(4, attn_type=input_attn)
        self.rgb_post_attn    = PostEncoderAttention(self.chs, attn_type=post_encoder_attn)
        self.aux_post_attn    = PostEncoderAttention(self.chs, attn_type=post_encoder_attn)
        self.dec_output_attn  = DecoderOutputAttention(fpn_channels, attn_type=decoder_output_attn)
        self.fusion   = build_fusion(fusion_name, self.chs)
        self.decoder  = HybridSegFormerUNetPPDecoder(self.chs, fpn_channels)
        self.head     = nn.Conv2d(fpn_channels, 1, kernel_size=1)
        self.img_size = img_size

    def _encode_rgb(self, rgb):
        feats = to_nchw(self.enc_rgb(self.rgb_input_attn(rgb)), self.chs)
        return self.rgb_post_attn(feats)

    def _encode_aux(self, aux4):
        feats = to_nchw(self.enc_aux(self.aux_input_attn(aux4)), self.chs)
        return self.aux_post_attn(feats)

    def forward(self, rgb, aux4):
        feats = self.fusion(self._encode_rgb(rgb), self._encode_aux(aux4))
        x, _ = self.decoder(feats)
        x = self.dec_output_attn(x)
        return F.interpolate(self.head(x), size=(self.img_size, self.img_size),
                             mode='bilinear', align_corners=False)

print('All model classes defined OK.')

In [ ]:
# ============================================================
# Cell 5 — ★ v6-DA InferenceDataset (per-image norm) + load_fold_models
# ============================================================

class InferenceDataset(Dataset):
    """Load phase2 .tif files with v6-DA per-image percentile normalization."""
    def __init__(self, image_dir, img_size, band_indices,
                 means, stds, rgb_indices, aux_indices):
        self.paths = sorted(Path(image_dir).glob('*.tif')) + sorted(Path(image_dir).glob('*.TIF'))
        self.img_size     = img_size
        self.band_indices = band_indices
        self.means = np.array(means, dtype=np.float32)
        self.stds  = np.array(stds,  dtype=np.float32)
        self.rgb_idx = np.array(rgb_indices, dtype=int)
        self.aux_idx = np.array(aux_indices, dtype=int)
        print(f'[InferenceDataset] {len(self.paths)} files found in {image_dir}')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        with rasterio.open(str(p)) as src:
            arr = src.read(self.band_indices).astype(np.float32)  # (7, H, W)

        # ★ v6-DA: Per-image percentile normalization (domain-invariant)
        arr = normalize_bands_per_image(arr)

        # Resize to IMG_SIZE if needed
        _, H, W = arr.shape
        if H != self.img_size or W != self.img_size:
            arr = np.stack([
                cv2.resize(arr[c], (self.img_size, self.img_size),
                           interpolation=cv2.INTER_LINEAR)
                for c in range(arr.shape[0])
            ], axis=0)

        # z-score standardize RGB and AUX separately (matches training _standardize)
        def _std(x_chw, indices):
            m = self.means[indices][:, None, None]
            s = self.stds[indices][:, None, None]
            return (x_chw - m) / s

        rgb_arr = _std(arr[self.rgb_idx], self.rgb_idx)   # (3, H, W)
        aux_arr = _std(arr[self.aux_idx], self.aux_idx)   # (4, H, W)

        return torch.from_numpy(rgb_arr), torch.from_numpy(aux_arr), p.name


def load_fold_models(ckpt_dir, fusion_name, num_folds, device):
    """Load all fold checkpoints for one fusion. Returns list of eval-mode models."""
    models = []
    ckpt_dir = Path(ckpt_dir)
    for fold in range(1, num_folds + 1):
        ckpt_path = ckpt_dir / f'fold{fold}_best.pt'
        if not ckpt_path.exists():
            print(f'  [WARNING] {ckpt_path} not found — skipping fold {fold}')
            continue
        model = DualSwinFusionSeg(
            encoder_name        = ENCODER_NAME,
            pretrained          = False,
            img_size            = IMG_SIZE,
            fpn_channels        = FPN_CHANNELS,
            fusion_name         = fusion_name,
            decoder_name        = DECODER_NAME,
            input_attn          = INPUT_ATTN,
            post_encoder_attn   = POST_ENCODER_ATTN,
            intra_encoder_attn  = INTRA_ENCODER_ATTN,
            decoder_output_attn = DECODER_OUTPUT_ATTN,
        ).to(device)
        ckpt  = torch.load(str(ckpt_path), map_location=device)
        state = ckpt.get('model', ckpt.get('model_state', ckpt.get('state_dict', ckpt)))
        miss, unexp = model.load_state_dict(state, strict=True)
        if miss:  print(f'  fold{fold} missing:    {miss[:3]}')
        if unexp: print(f'  fold{fold} unexpected: {unexp[:3]}')
        model.eval()
        bm = ckpt.get('best_metrics', {})
        val_iou = bm.get('miou', bm.get('val_miou', ckpt.get('best_miou', None)))
        print(f'  [{fusion_name}] fold{fold} loaded — val_mIoU: {val_iou:.4f}' if val_iou else
              f'  [{fusion_name}] fold{fold} loaded')
        models.append(model)
    return models


# ---- Build dataset & loader ----
# ★ v6-DA: No STATS_LOW / STATS_HIGH needed — per-image normalization is self-contained
test_dataset = InferenceDataset(
    image_dir   = TEST_DATA_DIR,
    img_size    = IMG_SIZE,
    band_indices= BAND_INDICES_ALL,
    means       = MEANS,
    stds        = STDS,
    rgb_indices = RGB_INDICES,
    aux_indices = AUX_INDICES,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == 'cuda'),
)
print(f'Test samples: {len(test_dataset)} | Batches: {len(test_loader)}')

# ---- Load all fold models for all fusions ----
all_models = []
for fusion_name in FUSIONS:
    print(f'\nLoading {NUM_FOLDS} folds for fusion={fusion_name} ...')
    fold_models = load_fold_models(CKPT_DIRS[fusion_name], fusion_name, NUM_FOLDS, DEVICE)
    all_models.extend(fold_models)

print(f'\nTotal models loaded: {len(all_models)} ({len(FUSIONS)} fusions × {NUM_FOLDS} folds)')
assert len(all_models) == NUM_FOLDS * len(FUSIONS), \
    f'Expected {NUM_FOLDS * len(FUSIONS)} models, got {len(all_models)} — some folds failed to load!'

# ---- Quick sanity check: first batch through first model ----
_rgb0, _aux0, _names0 = next(iter(test_loader))
print(f'\n--- Input sanity check (first batch) ---')
print(f'  RGB tensor: shape={_rgb0.shape} min={_rgb0.min():.3f} max={_rgb0.max():.3f} mean={_rgb0.mean():.3f}')
print(f'  AUX tensor: shape={_aux0.shape} min={_aux0.min():.3f} max={_aux0.max():.3f} mean={_aux0.mean():.3f}')
with torch.no_grad():
    _logits0 = all_models[0](_rgb0.to(DEVICE), _aux0.to(DEVICE))
    _probs0  = torch.sigmoid(_logits0)
    print(f'  Raw logits:  min={_logits0.min():.3f} max={_logits0.max():.3f} mean={_logits0.mean():.3f}')
    print(f'  Probs:       min={_probs0.min():.3f} max={_probs0.max():.3f} mean={_probs0.mean():.3f}')
    print(f'  Fraction > 0.5: {(_probs0 > 0.5).float().mean():.4f}')
del _rgb0, _aux0, _logits0, _probs0

In [ ]:
# ============================================================
# Cell 6 — TTA + Ensemble Inference → submission ZIP
# ============================================================

def _write_mask_tiff(mask01, out_path, height=128, width=128):
    """Write binary mask as GeoTIFF at orig_size resolution."""
    if mask01.shape[0] != height or mask01.shape[1] != width:
        mask01 = cv2.resize(mask01.astype(np.uint8), (width, height),
                            interpolation=cv2.INTER_NEAREST)
    with rasterio.open(
        out_path, 'w', driver='GTiff',
        height=height, width=width,
        count=1, dtype=rasterio.uint8,
    ) as dst:
        dst.write(mask01.astype(np.uint8), 1)


@torch.no_grad()
def tta_predict(model, rgb, aux):
    """4-fold flip TTA: original + H-flip + V-flip + HV-flip."""
    transforms = [
        (lambda r, a: (r, a),                lambda p: p),
        (lambda r, a: (r.flip(-1), a.flip(-1)), lambda p: p.flip(-1)),
        (lambda r, a: (r.flip(-2), a.flip(-2)), lambda p: p.flip(-2)),
        (lambda r, a: (r.flip(-1).flip(-2), a.flip(-1).flip(-2)), lambda p: p.flip(-1).flip(-2)),
    ]
    total = None
    for tfm, inv in transforms:
        r, a = tfm(rgb, aux)
        logits = model(r, a)
        p = inv(torch.sigmoid(logits))
        total = p if total is None else total + p
    return total / len(transforms)


@torch.no_grad()
def ensemble_predict(models, loader, out_dir, thresh=0.5, orig_size=128, use_tta=True):
    """Average probabilities across all models, threshold, write .tif masks."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for m in models: m.eval()
    written = 0
    total_fg_pixels = 0
    total_pixels = 0
    for rgb, aux, names in tqdm(loader, desc='Ensemble+TTA inference'):
        rgb = rgb.to(DEVICE, non_blocking=True)
        aux = aux.to(DEVICE, non_blocking=True)
        prob = torch.zeros(rgb.shape[0], 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
        for model in models:
            if use_tta:
                prob += tta_predict(model, rgb, aux)
            else:
                prob += torch.sigmoid(model(rgb, aux))
        prob /= len(models)
        masks = (prob[:, 0] > thresh).cpu().numpy().astype(np.uint8)
        for mask, name in zip(masks, names):
            out_path = str(out_dir / name)
            _write_mask_tiff(mask, out_path, height=orig_size, width=orig_size)
            total_fg_pixels += int(mask.sum())
            total_pixels += int(mask.size)
            written += 1
    fg_frac = total_fg_pixels / max(total_pixels, 1)
    print(f'[ensemble_predict] Wrote {written} mask TIFs → {out_dir}')
    print(f'  Total foreground pixels: {total_fg_pixels:,} / {total_pixels:,} = {fg_frac:.4%}')
    print(f'  Mean fg pixels/mask: {total_fg_pixels / max(written, 1):.1f}')
    print(f'  ★ For reference, v3 submission had mean ~4330 fg pixels/mask, {1195024 / (276 * 128 * 128):.4%} fg fraction')
    return out_dir


def zip_predictions(pred_dir, zip_path):
    pred_dir = Path(pred_dir)
    tifs = sorted(pred_dir.glob('*.tif')) + sorted(pred_dir.glob('*.TIF'))
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for t in tifs:
            zf.write(t, arcname=t.name)
    print(f'[zip_predictions] {len(tifs)} TIFs → {zip_path}')


# ---- Run inference ----
pred_dir = OUTPUT_DIR / 'predictions'
zip_path = str(OUTPUT_DIR / 'phase2_submission_DA.zip')

ensemble_predict(
    models   = all_models,
    loader   = test_loader,
    out_dir  = str(pred_dir),
    thresh   = THRESH,
    orig_size= ORIG_SIZE,
    use_tta  = USE_TTA,
)

zip_predictions(pred_dir, zip_path)

# Final diagnostic: zip size
_zip_size = os.path.getsize(zip_path)
print(f'\n✓ Submission ready: {zip_path}')
print(f'  ZIP size: {_zip_size:,} bytes ({_zip_size / 1024:.1f} KB)')
print(f'  ★ For reference, v3 submission_v3_inference.zip was ~179 KB')
if _zip_size < 80_000:
    print(f'  ⚠ WARNING: ZIP is unusually small! Check prediction diagnostics above.')
    print(f'  Possible causes: wrong normalization stats, wrong TEST_DATA_DIR, model not loaded correctly.')

In [ ]:
# ============================================================
# Cell 7 — Visualize sample predictions
# ============================================================
import matplotlib.pyplot as plt

pred_dir = OUTPUT_DIR / 'predictions'
pred_tifs = sorted(pred_dir.glob('*.tif'))
src_tifs  = sorted(Path(TEST_DATA_DIR).glob('*.tif'))

num_show = min(6, len(pred_tifs))
fig, axes = plt.subplots(num_show, 2, figsize=(8, 3 * num_show))
if num_show == 1:
    axes = [axes]

for i in range(num_show):
    pred_path = pred_tifs[i]
    # Try to find matching source image
    src_matches = [s for s in src_tifs if s.stem == pred_path.stem]
    ax_img, ax_mask = axes[i]

    # Show RGB composite from source
    if src_matches:
        with rasterio.open(str(src_matches[0])) as src:
            # RGB = bands 5,6,7 (1-indexed) → array position 4,5,6 (0-indexed)
            r = src.read(5).astype(np.float32)
            g = src.read(6).astype(np.float32)
            b = src.read(7).astype(np.float32)
        def norm_ch(x): return (x - x.min()) / (x.max() - x.min() + 1e-6)
        rgb_img = np.stack([norm_ch(r), norm_ch(g), norm_ch(b)], axis=-1)
        ax_img.imshow(rgb_img)
        ax_img.set_title(f'RGB — {pred_path.name}', fontsize=8)
    else:
        ax_img.text(0.5, 0.5, 'Source not found', ha='center', va='center')
    ax_img.axis('off')

    with rasterio.open(str(pred_path)) as pf:
        mask = pf.read(1)
    ax_mask.imshow(mask, cmap='gray', vmin=0, vmax=1)
    ax_mask.set_title(f'Pred mask (fg={mask.mean():.3f})', fontsize=8)
    ax_mask.axis('off')

plt.suptitle(f'Phase 2 Predictions — {len(pred_tifs)} tiles total\n'
             f'Ensemble: {len(all_models)} models ({len(FUSIONS)} fusions × {NUM_FOLDS} folds)',
             fontsize=10)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'prediction_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Sample visualization saved → {OUTPUT_DIR / "prediction_samples.png"}')

## v6-DA Summary — Per-Image Normalization

| | |
|---|---|
| **Model** | `DualSwinFusionSeg` — Hybrid SegFormer×UNet++ decoder |
| **Encoder** | `swinv2_small_window8_256` (dual: RGB 3ch + AUX 4ch) |
| **Fusion** | `concat_eca` |
| **Attention** | input=eca, intra_enc=se, dec_out=cbam |
| **Normalization** | ★ Per-image percentile (P1/P99 per band per image) → [0,1] → z-score |
| **Stats** | Loaded from `norm_stats_v6_DA.json` (z-score mean/std only) |
| **Ensemble** | 5 folds × 4-fold TTA |
| **Output** | `phase2_inference_DA_output/phase2_submission_DA.zip` |

**Domain shift handled**: DEM, B4, B5, B6 absolute value differences between train/test domains
are neutralized by normalizing each image to its own percentile range.